# Chronic Kidney Disease Prediction

## Problem Statement

The objective of this project is to develop a machine learning model that predicts whether a patient has **Chronic Kidney Disease (CKD)** based on clinical and laboratory parameters.

The target variable is `classification`:
- `yes` = CKD
- `no` = Not CKD

Multiple machine learning classification algorithms are trained, evaluated, and hyperparameter-tuned using `GridSearchCV`. The final model is selected based on the evaluation results, with particular importance given to **Recall**, because missing a CKD case (false negative) is important in a screening-oriented prediction problem.

> This is an academic machine-learning project and is not a clinically validated diagnostic system.


## 1. Import Required Libraries


In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

print("Libraries imported successfully.")


Libraries imported successfully.


## 2. Load the Dataset

Update the path below if your CSV file is stored somewhere else.


In [2]:
DATA_PATH = "../data/CKD.csv"

dataset = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Dataset shape:", dataset.shape)
dataset.head()


Dataset loaded successfully.
Dataset shape: (399, 25)


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,2.0,76.459948,c,3.0,0.0,normal,abnormal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,yes,no,yes
1,3.0,76.459948,c,2.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,34.000000,12300.000000,4.705597,no,no,no,yes,poor,no,yes
2,4.0,76.459948,a,1.0,0.0,normal,normal,notpresent,notpresent,99.000000,...,34.000000,8408.191126,4.705597,no,no,no,yes,poor,no,yes
3,5.0,76.459948,d,1.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,poor,yes,yes
4,5.0,50.000000,c,0.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,36.000000,12400.000000,4.705597,no,no,no,yes,poor,no,yes


## 3. Basic Dataset Information


In [3]:
print("Number of rows   :", dataset.shape[0])
print("Number of columns:", dataset.shape[1])

print("\nColumn names:")
print(dataset.columns.tolist())

print("\nData types:")
print(dataset.dtypes)


Number of rows   : 399
Number of columns: 25

Column names:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hrmo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'classification']

Data types:
age               float64
bp                float64
sg                 object
al                float64
su                float64
rbc                object
pc                 object
pcc                object
ba                 object
bgr               float64
bu                float64
sc                float64
sod               float64
pot               float64
hrmo              float64
pcv               float64
wc                float64
rc                float64
htn                object
dm                 object
cad                object
appet              object
pe                 object
ane                object
classification     object
dtype: object


## 3.1 Missing Values


In [4]:
print("Missing values by column:")
print(dataset.isnull().sum())

print("\nTotal missing values:", dataset.isnull().sum().sum())


Missing values by column:
age               0
bp                0
sg                0
al                0
su                0
rbc               0
pc                0
pcc               0
ba                0
bgr               0
bu                0
sc                0
sod               0
pot               0
hrmo              0
pcv               0
wc                0
rc                0
htn               0
dm                0
cad               0
appet             0
pe                0
ane               0
classification    0
dtype: int64

Total missing values: 0


## 3.2 Duplicate Rows


In [5]:
duplicate_count = dataset.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)


Number of duplicate rows: 0


## 3.3 Target Distribution


In [6]:
print(dataset["classification"].value_counts())

print("\nPercentage distribution:")
print(
    (dataset["classification"].value_counts(normalize=True) * 100)
    .round(2)
)


classification
yes    249
no     150
Name: count, dtype: int64

Percentage distribution:
classification
yes    62.41
no     37.59
Name: proportion, dtype: float64


## 4. Exploratory Data Analysis

The dataset contains both numerical and categorical features. The categorical values are inspected before preprocessing.


In [7]:
categorical_columns = [
    "sg", "rbc", "pc", "pcc", "ba",
    "htn", "dm", "cad", "appet", "pe", "ane"
]

print("Categorical columns:")
print(categorical_columns)

for column in categorical_columns:
    print(f"\n{column}:")
    print(dataset[column].unique())


Categorical columns:
['sg', 'rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']

sg:
['c' 'a' 'd' 'b' 'e']

rbc:
['normal' 'abnormal']

pc:
['abnormal' 'normal']

pcc:
['notpresent' 'present']

ba:
['notpresent' 'present']

htn:
['no' 'yes']

dm:
['no' 'yes']

cad:
['no' 'yes']

appet:
['yes' 'poor']

pe:
['yes' 'poor']

ane:
['no' 'yes']


## 5. Data Preprocessing

The preprocessing steps are:

1. Separate the target from the input features.
2. Convert categorical variables to numerical variables using one-hot encoding.
3. Encode the target as `0` and `1`.
4. Split the data into training and testing sets using stratification.
5. Standardize the training features for Logistic Regression, KNN, and SVM.
6. Keep the test set untouched for final evaluation.


### 5.1 Separate Features and Target


In [8]:
X = dataset.drop("classification", axis=1)
y = dataset["classification"].map({"no": 0, "yes": 1})

print("Original X shape:", X.shape)
print("y shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())


Original X shape: (399, 24)
y shape: (399,)

Target distribution:
classification
1    249
0    150
Name: count, dtype: int64


### 5.2 One-Hot Encode Categorical Features


In [9]:
X_encoded = pd.get_dummies(
    X,
    columns=categorical_columns,
    drop_first=True
)

print("Encoded X shape:", X_encoded.shape)
print("\nEncoded columns:")
print(X_encoded.columns.tolist())


Encoded X shape: (399, 27)

Encoded columns:
['age', 'bp', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hrmo', 'pcv', 'wc', 'rc', 'sg_b', 'sg_c', 'sg_d', 'sg_e', 'rbc_normal', 'pc_normal', 'pcc_present', 'ba_present', 'htn_yes', 'dm_yes', 'cad_yes', 'appet_yes', 'pe_yes', 'ane_yes']


### 5.3 Train-Test Split


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)


X_train shape: (319, 27)
X_test shape : (80, 27)
y_train shape: (319,)
y_test shape : (80,)


### 5.4 Feature Scaling

Tree-based algorithms do not require feature scaling. Logistic Regression, KNN, and SVM are trained using standardized features.


In [11]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape :", X_test_scaled.shape)


X_train_scaled shape: (319, 27)
X_test_scaled shape : (80, 27)


## 6. Model Evaluation Function

This function calculates all required evaluation metrics so the same method is used consistently for every model.


In [12]:
def evaluate_model(model_name, model, X_eval, y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    if hasattr(model, "predict_proba"):
        y_probability = model.predict_proba(X_eval)[:, 1]
    else:
        y_probability = model.decision_function(X_eval)

    roc_auc = roc_auc_score(y_true, y_probability)

    print(f"{model_name} Results")
    print("-" * 35)
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"ROC-AUC  : {roc_auc:.4f}")

    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    }


# 7. Baseline Machine Learning Models

Before hyperparameter tuning, baseline models are trained so that their performance can be compared with the tuned versions.


## 7.1 Logistic Regression


In [13]:
logistic_model = LogisticRegression(
    max_iter=2000,
    random_state=42
)

logistic_model.fit(X_train_scaled, y_train)
y_pred_lr = logistic_model.predict(X_test_scaled)

lr_result = evaluate_model(
    "Logistic Regression",
    logistic_model,
    X_test_scaled,
    y_test,
    y_pred_lr
)


Logistic Regression Results
-----------------------------------
Accuracy : 0.9875
Precision: 1.0000
Recall   : 0.9800
F1 Score : 0.9899
ROC-AUC  : 1.0000


## 7.2 Decision Tree


In [14]:
decision_tree_model = DecisionTreeClassifier(
    random_state=42
)

decision_tree_model.fit(X_train, y_train)
y_pred_dt = decision_tree_model.predict(X_test)

dt_result = evaluate_model(
    "Decision Tree",
    decision_tree_model,
    X_test,
    y_test,
    y_pred_dt
)


Decision Tree Results
-----------------------------------
Accuracy : 0.9875
Precision: 0.9804
Recall   : 1.0000
F1 Score : 0.9901
ROC-AUC  : 0.9833


## 7.3 Random Forest


In [15]:
random_forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

random_forest_model.fit(X_train, y_train)
y_pred_rf = random_forest_model.predict(X_test)

rf_result = evaluate_model(
    "Random Forest",
    random_forest_model,
    X_test,
    y_test,
    y_pred_rf
)


Random Forest Results
-----------------------------------
Accuracy : 0.9875
Precision: 1.0000
Recall   : 0.9800
F1 Score : 0.9899
ROC-AUC  : 0.9993


## 7.4 K-Nearest Neighbors (KNN)


In [16]:
knn_model = KNeighborsClassifier(
    n_neighbors=5
)

knn_model.fit(X_train_scaled, y_train)
y_pred_knn = knn_model.predict(X_test_scaled)

knn_result = evaluate_model(
    "KNN",
    knn_model,
    X_test_scaled,
    y_test,
    y_pred_knn
)


KNN Results
-----------------------------------
Accuracy : 0.9625
Precision: 1.0000
Recall   : 0.9400
F1 Score : 0.9691
ROC-AUC  : 0.9993


## 7.5 Support Vector Machine (SVM)


In [17]:
svm_model = SVC(
    kernel="rbf",
    probability=True,
    random_state=42
)

svm_model.fit(X_train_scaled, y_train)
y_pred_svm = svm_model.predict(X_test_scaled)

svm_result = evaluate_model(
    "SVM",
    svm_model,
    X_test_scaled,
    y_test,
    y_pred_svm
)


SVM Results
-----------------------------------
Accuracy : 0.9875
Precision: 1.0000
Recall   : 0.9800
F1 Score : 0.9899
ROC-AUC  : 1.0000


## 7.6 Gradient Boosting


In [18]:
gradient_boosting_model = GradientBoostingClassifier(
    random_state=42
)

gradient_boosting_model.fit(X_train, y_train)
y_pred_gb = gradient_boosting_model.predict(X_test)

gb_result = evaluate_model(
    "Gradient Boosting",
    gradient_boosting_model,
    X_test,
    y_test,
    y_pred_gb
)


Gradient Boosting Results
-----------------------------------
Accuracy : 0.9875
Precision: 1.0000
Recall   : 0.9800
F1 Score : 0.9899
ROC-AUC  : 0.9993


## 7.7 Baseline Model Comparison


In [19]:
baseline_results = pd.DataFrame([
    lr_result,
    dt_result,
    rf_result,
    knn_result,
    svm_result,
    gb_result
])

baseline_results.round(4)


,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.9875,1.0000,0.98,0.9899,1.0000
1,Decision Tree,0.9875,0.9804,1.00,0.9901,0.9833
2,Random Forest,0.9875,1.0000,0.98,0.9899,0.9993
3,KNN,0.9625,1.0000,0.94,0.9691,0.9993
4,SVM,0.9875,1.0000,0.98,0.9899,1.0000
5,Gradient Boosting,0.9875,1.0000,0.98,0.9899,0.9993


# 8. Hyperparameter Tuning Using GridSearchCV

GridSearchCV is used to identify suitable hyperparameters using 5-fold cross-validation.

### Important methodological improvement

For Logistic Regression, KNN, and SVM, scaling is placed **inside a Pipeline**. This ensures that the scaler is fitted separately within each cross-validation training fold and prevents information leakage between CV folds.

The test set remains completely untouched until final evaluation.

### Model-selection strategy

Because CKD screening should minimize missed CKD cases, **Recall is the primary model-selection metric**. F1 Score and ROC-AUC are used as secondary indicators.


In [20]:
from sklearn.pipeline import Pipeline

CV = 5

scoring = {
    "Accuracy": "accuracy",
    "Precision": "precision",
    "Recall": "recall",
    "F1": "f1",
    "ROC-AUC": "roc_auc"
}

print("Cross-validation folds:", CV)
print("Primary GridSearch metric: Recall")


Cross-validation folds: 5
Primary GridSearch metric: Recall


## 8.1 Logistic Regression - GridSearchCV


In [21]:
lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, random_state=42))
])

lr_param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__solver": ["liblinear", "lbfgs"]
}

lr_grid = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=lr_param_grid,
    scoring=scoring,
    refit="Recall",
    cv=CV,
    n_jobs=-1,
    return_train_score=False
)

lr_grid.fit(X_train, y_train)

best_lr = lr_grid.best_estimator_
y_pred_lr_tuned = best_lr.predict(X_test)

tuned_lr_result = evaluate_model(
    "Tuned Logistic Regression",
    best_lr,
    X_test,
    y_test,
    y_pred_lr_tuned
)

print("\nBest parameters:", lr_grid.best_params_)
print(f"Best CV Recall: {lr_grid.best_score_:.4f}")


Tuned Logistic Regression Results
-----------------------------------
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1 Score : 1.0000
ROC-AUC  : 1.0000

Best parameters: {'model__C': 0.01, 'model__solver': 'lbfgs'}
Best CV Recall: 0.9849


## 8.2 Decision Tree - GridSearchCV


In [22]:
dt_param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [None, 3, 5, 7, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

dt_grid = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=dt_param_grid,
    scoring=scoring,
    refit="Recall",
    cv=CV,
    n_jobs=-1,
    return_train_score=False
)

dt_grid.fit(X_train, y_train)

best_dt = dt_grid.best_estimator_
y_pred_dt_tuned = best_dt.predict(X_test)

tuned_dt_result = evaluate_model(
    "Tuned Decision Tree",
    best_dt,
    X_test,
    y_test,
    y_pred_dt_tuned
)

print("\nBest parameters:", dt_grid.best_params_)
print(f"Best CV Recall: {dt_grid.best_score_:.4f}")


Tuned Decision Tree Results
-----------------------------------
Accuracy : 0.9750
Precision: 0.9800
Recall   : 0.9800
F1 Score : 0.9800
ROC-AUC  : 0.9993

Best parameters: {'criterion': 'gini', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 10}
Best CV Recall: 0.9546


## 8.3 Random Forest - GridSearchCV


In [23]:
rf_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

rf_grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=rf_param_grid,
    scoring=scoring,
    refit="Recall",
    cv=CV,
    n_jobs=-1,
    return_train_score=False
)

rf_grid.fit(X_train, y_train)

best_rf = rf_grid.best_estimator_
y_pred_rf_tuned = best_rf.predict(X_test)

tuned_rf_result = evaluate_model(
    "Tuned Random Forest",
    best_rf,
    X_test,
    y_test,
    y_pred_rf_tuned
)

print("\nBest parameters:", rf_grid.best_params_)
print(f"Best CV Recall: {rf_grid.best_score_:.4f}")


Tuned Random Forest Results
-----------------------------------
Accuracy : 0.9875
Precision: 1.0000
Recall   : 0.9800
F1 Score : 0.9899
ROC-AUC  : 0.9993

Best parameters: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
Best CV Recall: 0.9899


## 8.4 KNN - GridSearchCV


In [24]:
knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier())
])

knn_param_grid = {
    "model__n_neighbors": [3, 5, 7, 9, 11, 15],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2]
}

knn_grid = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=knn_param_grid,
    scoring=scoring,
    refit="Recall",
    cv=CV,
    n_jobs=-1,
    return_train_score=False
)

knn_grid.fit(X_train, y_train)

best_knn = knn_grid.best_estimator_
y_pred_knn_tuned = best_knn.predict(X_test)

tuned_knn_result = evaluate_model(
    "Tuned KNN",
    best_knn,
    X_test,
    y_test,
    y_pred_knn_tuned
)

print("\nBest parameters:", knn_grid.best_params_)
print(f"Best CV Recall: {knn_grid.best_score_:.4f}")


Tuned KNN Results
-----------------------------------
Accuracy : 0.9625
Precision: 1.0000
Recall   : 0.9400
F1 Score : 0.9691
ROC-AUC  : 0.9993

Best parameters: {'model__n_neighbors': 5, 'model__p': 2, 'model__weights': 'uniform'}
Best CV Recall: 0.9647


## 8.5 SVM - GridSearchCV


In [25]:
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(probability=True, random_state=42))
])

svm_param_grid = [
    {
        "model__C": [0.1, 1, 10, 100],
        "model__kernel": ["linear"]
    },
    {
        "model__C": [0.1, 1, 10, 100],
        "model__kernel": ["rbf"],
        "model__gamma": ["scale", "auto"]
    }
]

svm_grid = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=svm_param_grid,
    scoring=scoring,
    refit="Recall",
    cv=CV,
    n_jobs=-1,
    return_train_score=False
)

svm_grid.fit(X_train, y_train)

best_svm = svm_grid.best_estimator_
y_pred_svm_tuned = best_svm.predict(X_test)

tuned_svm_result = evaluate_model(
    "Tuned SVM",
    best_svm,
    X_test,
    y_test,
    y_pred_svm_tuned
)

print("\nBest parameters:", svm_grid.best_params_)
print(f"Best CV Recall: {svm_grid.best_score_:.4f}")


Tuned SVM Results
-----------------------------------
Accuracy : 0.9875
Precision: 1.0000
Recall   : 0.9800
F1 Score : 0.9899
ROC-AUC  : 1.0000

Best parameters: {'model__C': 1, 'model__gamma': 'scale', 'model__kernel': 'rbf'}
Best CV Recall: 0.9800


## 8.6 Gradient Boosting - GridSearchCV


In [26]:
gb_param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [1, 3, 5],
    "subsample": [0.8, 1.0]
}

gb_grid = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_grid=gb_param_grid,
    scoring=scoring,
    refit="Recall",
    cv=CV,
    n_jobs=-1,
    return_train_score=False
)

gb_grid.fit(X_train, y_train)

best_gb = gb_grid.best_estimator_
y_pred_gb_tuned = best_gb.predict(X_test)

tuned_gb_result = evaluate_model(
    "Tuned Gradient Boosting",
    best_gb,
    X_test,
    y_test,
    y_pred_gb_tuned
)

print("\nBest parameters:", gb_grid.best_params_)
print(f"Best CV Recall: {gb_grid.best_score_:.4f}")


Tuned Gradient Boosting Results
-----------------------------------
Accuracy : 0.9875
Precision: 1.0000
Recall   : 0.9800
F1 Score : 0.9899
ROC-AUC  : 1.0000

Best parameters: {'learning_rate': 0.1, 'max_depth': 1, 'n_estimators': 200, 'subsample': 0.8}
Best CV Recall: 0.9847


# 9. Tuned Model Comparison

This table reports the final test-set performance of the tuned models.

**Important:** These test results are for final evaluation. Model selection is based primarily on cross-validation performance, not by repeatedly choosing the best model from the test set.


In [27]:
tuned_results = pd.DataFrame([
    tuned_lr_result,
    tuned_dt_result,
    tuned_rf_result,
    tuned_knn_result,
    tuned_svm_result,
    tuned_gb_result
])

tuned_results.round(4)


,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Tuned Logistic Regression,1.0000,1.00,1.00,1.0000,1.0000
1,Tuned Decision Tree,0.9750,0.98,0.98,0.9800,0.9993
2,Tuned Random Forest,0.9875,1.00,0.98,0.9899,0.9993
3,Tuned KNN,0.9625,1.00,0.94,0.9691,0.9993
4,Tuned SVM,0.9875,1.00,0.98,0.9899,1.0000
5,Tuned Gradient Boosting,0.9875,1.00,0.98,0.9899,1.0000


## 9.1 Cross-Validation Results and Best Hyperparameters


In [28]:
grid_objects = {
    "Logistic Regression": lr_grid,
    "Decision Tree": dt_grid,
    "Random Forest": rf_grid,
    "KNN": knn_grid,
    "SVM": svm_grid,
    "Gradient Boosting": gb_grid
}

cv_summary = []

for name, grid in grid_objects.items():
    best_index = grid.best_index_
    cv_summary.append({
        "Model": name,
        "Best Parameters": grid.best_params_,
        "Mean CV Recall": grid.cv_results_["mean_test_Recall"][best_index],
        "Mean CV Precision": grid.cv_results_["mean_test_Precision"][best_index],
        "Mean CV F1": grid.cv_results_["mean_test_F1"][best_index],
        "Mean CV ROC-AUC": grid.cv_results_["mean_test_ROC-AUC"][best_index],
        "Mean CV Accuracy": grid.cv_results_["mean_test_Accuracy"][best_index]
    })

cv_results = pd.DataFrame(cv_summary)

cv_results = cv_results.sort_values(
    by=["Mean CV Recall", "Mean CV F1", "Mean CV ROC-AUC", "Mean CV Accuracy"],
    ascending=False
).reset_index(drop=True)

cv_results.round(4)


,Model,Best Parameters,Mean CV Recall,Mean CV Precision,Mean CV F1,Mean CV ROC-AUC,Mean CV Accuracy
0,Random Forest,"{'max_depth': None, 'min_samples_leaf': 1, 'mi...",0.9899,0.9900,0.9899,0.9972,0.9874
1,Logistic Regression,"{'model__C': 0.01, 'model__solver': 'lbfgs'}",0.9849,0.9951,0.9898,0.9994,0.9875
2,Gradient Boosting,"{'learning_rate': 0.1, 'max_depth': 1, 'n_esti...",0.9847,0.9847,0.9847,0.9979,0.9811
3,SVM,"{'model__C': 1, 'model__gamma': 'scale', 'mode...",0.9800,0.9951,0.9873,0.9998,0.9844
4,KNN,"{'model__n_neighbors': 5, 'model__p': 2, 'mode...",0.9647,1.0000,0.9820,0.9912,0.9780
5,Decision Tree,"{'criterion': 'gini', 'max_depth': None, 'min_...",0.9546,0.9747,0.9642,0.9672,0.9560


# 10. Baseline vs Tuned Comparison


In [29]:
baseline_report = baseline_results.copy()
baseline_report["Version"] = "Baseline"

tuned_report = tuned_results.copy()
tuned_report["Version"] = "Tuned"

comparison_results = pd.concat(
    [baseline_report, tuned_report],
    ignore_index=True
)

comparison_results.round(4)


,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC,Version
0,Logistic Regression,0.9875,1.0000,0.98,0.9899,1.0000,Baseline
1,Decision Tree,0.9875,0.9804,1.00,0.9901,0.9833,Baseline
2,Random Forest,0.9875,1.0000,0.98,0.9899,0.9993,Baseline
3,KNN,0.9625,1.0000,0.94,0.9691,0.9993,Baseline
4,SVM,0.9875,1.0000,0.98,0.9899,1.0000,Baseline
5,Gradient Boosting,0.9875,1.0000,0.98,0.9899,0.9993,Baseline
6,Tuned Logistic Regression,1.0000,1.0000,1.00,1.0000,1.0000,Tuned
7,Tuned Decision Tree,0.9750,0.9800,0.98,0.9800,0.9993,Tuned
8,Tuned Random Forest,0.9875,1.0000,0.98,0.9899,0.9993,Tuned
9,Tuned KNN,0.9625,1.0000,0.94,0.9691,0.9993,Tuned


# 11. Confusion Matrices for Tuned Models


In [30]:
tuned_predictions = {
    "Tuned Logistic Regression": y_pred_lr_tuned,
    "Tuned Decision Tree": y_pred_dt_tuned,
    "Tuned Random Forest": y_pred_rf_tuned,
    "Tuned KNN": y_pred_knn_tuned,
    "Tuned SVM": y_pred_svm_tuned,
    "Tuned Gradient Boosting": y_pred_gb_tuned
}

confusion_summary = []

for model_name, predictions in tuned_predictions.items():
    cm = confusion_matrix(y_test, predictions)
    tn, fp, fn, tp = cm.ravel()

    confusion_summary.append({
        "Model": model_name,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })

    print(f"\n{model_name}")
    print(cm)

confusion_summary = pd.DataFrame(confusion_summary)
confusion_summary



Tuned Logistic Regression
[[30  0]
 [ 0 50]]

Tuned Decision Tree
[[29  1]
 [ 1 49]]

Tuned Random Forest
[[30  0]
 [ 1 49]]

Tuned KNN
[[30  0]
 [ 3 47]]

Tuned SVM
[[30  0]
 [ 1 49]]

Tuned Gradient Boosting
[[30  0]
 [ 1 49]]


,Model,TN,FP,FN,TP
0,Tuned Logistic Regression,30,0,0,50
1,Tuned Decision Tree,29,1,1,49
2,Tuned Random Forest,30,0,1,49
3,Tuned KNN,30,0,3,47
4,Tuned SVM,30,0,1,49
5,Tuned Gradient Boosting,30,0,1,49


# 12. Final Model Selection

### Selection rule

The final model is selected using **5-fold cross-validation Recall** as the primary criterion because false negatives are especially important for CKD screening.

Secondary criteria:
1. Mean CV F1 Score
2. Mean CV ROC-AUC
3. Mean CV Accuracy

Only after this selection is the chosen model's performance reported on the untouched test set.

This approach avoids choosing a model solely because it happened to perform best on the test split.


In [31]:
# Select the algorithm with the strongest cross-validation performance.
best_cv_model_name = cv_results.loc[0, "Model"]

final_model_map = {
    "Logistic Regression": best_lr,
    "Decision Tree": best_dt,
    "Random Forest": best_rf,
    "KNN": best_knn,
    "SVM": best_svm,
    "Gradient Boosting": best_gb
}

final_grid_map = {
    "Logistic Regression": lr_grid,
    "Decision Tree": dt_grid,
    "Random Forest": rf_grid,
    "KNN": knn_grid,
    "SVM": svm_grid,
    "Gradient Boosting": gb_grid
}

final_model = final_model_map[best_cv_model_name]
final_grid = final_grid_map[best_cv_model_name]

print("Selected final model:", best_cv_model_name)
print("Best hyperparameters:", final_grid.best_params_)
print(f"Mean CV Recall: {final_grid.best_score_:.4f}")


Selected final model: Random Forest
Best hyperparameters: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
Mean CV Recall: 0.9899


## 12.1 Final Model Test Performance


In [32]:
final_prediction_map = {
    "Logistic Regression": y_pred_lr_tuned,
    "Decision Tree": y_pred_dt_tuned,
    "Random Forest": y_pred_rf_tuned,
    "KNN": y_pred_knn_tuned,
    "SVM": y_pred_svm_tuned,
    "Gradient Boosting": y_pred_gb_tuned
}

final_predictions = final_prediction_map[best_cv_model_name]

final_test_result = evaluate_model(
    f"Final Model - {best_cv_model_name}",
    final_model,
    X_test,
    y_test,
    final_predictions
)

final_cm = confusion_matrix(y_test, final_predictions)

print("\nFinal model confusion matrix:")
print(final_cm)


Final Model - Random Forest Results
-----------------------------------
Accuracy : 0.9875
Precision: 1.0000
Recall   : 0.9800
F1 Score : 0.9899
ROC-AUC  : 0.9993

Final model confusion matrix:
[[30  0]
 [ 1 49]]


## 12.2 Final Model Justification

The final model was selected after hyperparameter tuning using 5-fold GridSearchCV. Recall was used as the primary selection metric because the objective of Chronic Kidney Disease (CKD) prediction is to identify CKD cases while minimizing false-negative predictions. F1 Score, ROC-AUC, and Accuracy were considered as secondary evaluation metrics.

Based on the cross-validation results, the **Tuned Random Forest model** was selected as the final model because it provided the strongest overall performance according to the defined model-selection criteria.

The selected Random Forest model was evaluated on the previously untouched test dataset and achieved the following results:

* **Accuracy:** 98.75%
* **Precision:** 100.00%
* **Recall:** 98.00%
* **F1 Score:** 98.99%
* **ROC-AUC:** 99.93%

The confusion matrix for the final model was:

```text
[[30  0]
 [ 1 49]]
```

This means that the model correctly classified **30 non-CKD cases** and **49 CKD cases**. It produced **0 false positives** and **1 false negative** on the test dataset.

Therefore, the **Tuned Random Forest model** was selected as the final model for the Chronic Kidney Disease Prediction project. Its high Recall indicates that it successfully identified most CKD cases, while its high Precision indicates that the CKD predictions were highly reliable on the test dataset. The high F1 Score and ROC-AUC further demonstrate strong classification performance.

The test-set results provide the final evaluation of the selected model within this train-test experiment. Further validation using an independent clinical dataset would be required before considering real-world clinical deployment.

# 13. Export the Final Model

The final tuned model is exported together with the feature-column order.

For the selected Logistic Regression, KNN, or SVM model, the scaler is already part of the exported Pipeline. Therefore, deployment does **not** need a separate scaler transformation for those models.


In [33]:
os.makedirs("../models", exist_ok=True)

joblib.dump(
    final_model,
    "../models/final_ckd_model_gridsearch.pkl"
)

joblib.dump(
    X_encoded.columns.tolist(),
    "../models/ckd_feature_columns.pkl"
)

print("Final model exported successfully.")
print("Model:", best_cv_model_name)
print("Path: ../models/final_ckd_model_gridsearch.pkl")
print("Feature columns: ../models/ckd_feature_columns.pkl")


Final model exported successfully.
Model: Random Forest
Path: ../models/final_ckd_model_gridsearch.pkl
Feature columns: ../models/ckd_feature_columns.pkl


# 14. Final Model Deployment Test

The deployment section below loads the exported model and predicts CKD for a new patient.

The preprocessing is kept consistent with the training data. If the final model is Logistic Regression, KNN, or SVM, its scaler is automatically applied by the Pipeline.


In [34]:
loaded_model = joblib.load("../models/final_ckd_model_gridsearch.pkl")
loaded_feature_columns = joblib.load("../models/ckd_feature_columns.pkl")

print("Final model loaded successfully.")
print("Selected model:", best_cv_model_name)
print("Expected feature count:", len(loaded_feature_columns))


Final model loaded successfully.
Selected model: Random Forest
Expected feature count: 27


## 14.1 Example Patient Input


In [35]:
patient = {
    "age": 48,
    "bp": 80,
    "sg": "a",
    "al": 0,
    "su": 0,
    "rbc": "normal",
    "pc": "normal",
    "pcc": "notpresent",
    "ba": "notpresent",
    "bgr": 120,
    "bu": 40,
    "sc": 1.2,
    "sod": 140,
    "pot": 4.5,
    "hrmo": 13.5,
    "pcv": 40,
    "wc": 8000,
    "rc": 5.0,
    "htn": "no",
    "dm": "no",
    "cad": "no",
    "appet": "yes",
    "pe": "poor",
    "ane": "no"
}

patient_df = pd.DataFrame([patient])

patient_encoded = pd.get_dummies(
    patient_df,
    columns=categorical_columns,
    drop_first=True
)

patient_encoded = patient_encoded.reindex(
    columns=loaded_feature_columns,
    fill_value=0
)

print("Patient encoded shape:", patient_encoded.shape)


Patient encoded shape: (1, 27)


## 14.2 Predict CKD


In [36]:
prediction = loaded_model.predict(patient_encoded)

if hasattr(loaded_model, "predict_proba"):
    probabilities = loaded_model.predict_proba(patient_encoded)[0]
    not_ckd_probability = probabilities[0] * 100
    ckd_probability = probabilities[1] * 100
else:
    not_ckd_probability = None
    ckd_probability = None

result = "CKD" if prediction[0] == 1 else "Not CKD"

print("===================================")
print("       CKD PREDICTION RESULT")
print("===================================")
print("Prediction:", result)

if ckd_probability is not None:
    print(f"Probability of Not CKD: {not_ckd_probability:.2f}%")
    print(f"Probability of CKD:     {ckd_probability:.2f}%")

print("===================================")


       CKD PREDICTION RESULT
Prediction: Not CKD
Probability of Not CKD: 77.00%
Probability of CKD:     23.00%


# 15. Conclusion

This project developed a CKD classification model using 24 input features from a dataset containing 399 records and 25 columns including the target.

Six machine-learning algorithms were evaluated:
- Logistic Regression
- Decision Tree
- Random Forest
- KNN
- SVM
- Gradient Boosting

GridSearchCV with 5-fold cross-validation was used for hyperparameter tuning. Recall was selected as the primary model-selection metric because false-negative CKD predictions are particularly important in a screening context.

The final model was selected using cross-validation performance and then evaluated on the untouched test set.

### Important interpretation

The dataset is relatively small, and extremely high test-set scores should not be interpreted as proof of clinical accuracy. Independent validation on a separate clinical dataset would be required before real-world use.

The notebook also exports the final tuned model for demonstration deployment.
